**Swiggy Data Analysis**

**1) Business Objective**

To assess and improve the quality, consistency, and completeness of the grocery product listing data by identifying anomalies, inconsistencies, missing information, duplicates, and pricing irregularities across products, categories, stores, and locations, thereby ensuring accurate product information and enhancing the customer shopping experience.

**2) Import Libraries**

In [1]:
import pandas as pd        # For data manipulation
import numpy as np      # For numerical computations
import seaborn as sns  # For advanced visualization
import matplotlib.pyplot as plt # For plotting
import re # For regular expressions

/usr/lib/python3/dist-packages/scipy/__init__.py:146: UserWarning: A NumPy version >=1.17.3 and <1.25.0 is required for this version of SciPy (detected version 1.26.4
  warnings.warn(f"A NumPy version >={np_minversion} and <{np_maxversion}"


**3) Import Dataset**

In [4]:
df = pd.read_csv("/home/user/Downloads/2026-07-08/swiggy_instamart_2026_07_07.csv")  #Reading Dataset from csv file

**4) Data Insights**

In [5]:
print("INFORMATION ABOUT DATASET")
print("--------------------------------")
df.info() 

INFORMATION ABOUT DATASET
--------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1177620 entries, 0 to 1177619
Data columns (total 31 columns):
 #   Column                   Non-Null Count    Dtype  
---  ------                   --------------    -----  
 0   crawl_date_and_time      1177620 non-null  object 
 1   platform_name            1177620 non-null  object 
 2   store_id                 1177620 non-null  int64  
 3   store_name_location      1177620 non-null  object 
 4   city                     1177620 non-null  object 
 5   pin_code                 1177620 non-null  int64  
 6   unique_id                1177620 non-null  object 
 7   product_id_upc_ean       1177620 non-null  object 
 8   department               1177620 non-null  object 
 9   category                 1177620 non-null  object 
 10  sub_category             1177620 non-null  object 
 11  product_type             0 non-null        float64
 12  brand_name               1177440 non-nu

In [6]:
print("\nDESCRIPTIVE STATISTICS OF DATASET")
print("--------------------------------")
df.describe().T.round(2) 


DESCRIPTIVE STATISTICS OF DATASET
--------------------------------


,count,mean,std,min,25%,50%,75%,max
store_id,1177620.0,1.368107e+06,8.606850e+04,867466.0,1385649.0,1393828.0,1399707.0,1.404909e+06
pin_code,1177620.0,4.535898e+05,1.615161e+05,110019.0,400006.0,500029.0,560024.0,8.340260e+05
product_type,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
product_description,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
package_type,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
net_quantity,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
item_mrp,1177620.0,4.915769e+08,6.616453e+10,1.0,110.0,225.0,440.0,8.906005e+12
item_selling_price,1177620.0,4.915768e+08,6.616453e+10,1.0,94.0,188.0,360.0,8.906005e+12
product_price,1177620.0,4.915768e+08,6.616453e+10,1.0,94.0,188.0,360.0,8.906005e+12
unit_price,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
print("\nDATASET SHAPE")
print("--------------------------------")
print("The shape =", df.shape)
num_rows, num_cols = df.shape
num_features = num_cols - 1
num_data = num_rows * num_cols

# Print the information about the dataset
print(f"Number of Rows: {num_rows}")
print(f"Number of Columns: {num_cols}")
print(f"Number of Features: {num_features}")
print(f"Number of All Data: {num_data}")


DATASET SHAPE
--------------------------------
The shape = (1177620, 31)
Number of Rows: 1177620
Number of Columns: 31
Number of Features: 30
Number of All Data: 36506220


In [8]:
print("COLUMNWISE DATA PROFILING")
print("--------------------------------")
summary = []

for col in df.columns:
    null_count = df[col].isnull().sum()
    null_percent = round((null_count / len(df)) * 100, 2)
    distinct_count = df[col].nunique(dropna=True)
    
    # Check if all non-null values are unique
    is_unique = "Yes" if df[col].is_unique and null_count == 0 else "No"


    summary.append({
        'column_name': col,
        'null_count': null_count,
        #'null_percent': null_percent,
        'distinct_count': distinct_count,
        'is_unique': is_unique
    })

summary_df = pd.DataFrame(summary)

# Display the summary
print(summary_df)

COLUMNWISE DATA PROFILING
--------------------------------
                column_name  null_count  distinct_count is_unique
0       crawl_date_and_time           0            8681        No
1             platform_name           0               1        No
2                  store_id           0             116        No
3       store_name_location           0             100        No
4                      city           0              38        No
5                  pin_code           0              95        No
6                 unique_id           0          953759        No
7        product_id_upc_ean           0           31263        No
8                department           0               9        No
9                  category           0              16        No
10             sub_category           0             148        No
11             product_type     1177620               0        No
12               brand_name         180            2921        No
13             pr

In [9]:
df.store_id.unique()
print(len(df.store_id.unique()))
print(len(df.store_name_location.unique()))

116
100


In [14]:
store_mapping = (
    df.groupby('store_name_location')['store_id']
      .nunique()
      .reset_index(name='unique_store_ids')
      .sort_values('unique_store_ids', ascending=False)
)

multi_store = store_mapping[store_mapping['unique_store_ids'] > 1]

print(multi_store)
print("\nNumber of store_name_location with multiple store_ids:", len(multi_store))

                                  store_name_location  unique_store_ids
99  Zepto KK Nagar, PV Rajamannar Salai, Opp. To R...                 2
24  CS Guest House, Mogappair East, 1/8, Bazaar Rd...                 2
78  Reliance Smart SuperStore, No 6/7, Ground Floo...                 2
77  Reddy Nilaya, 44, Appareddy Palya Extension Ro...                 2
58  MRPL R R Petroleum, JP Nagar 9th Phase, J. P. ...                 2
30  D-Mart, Dmart Yelahaka Chikka Bommasandra, Nor...                 2
76  Real Deepak Punjabi Dhaba, NAD Junction, Sanje...                 2
71  Piles Clinic In Indore | Piles Treatment | Fis...                 2
70  Pawan Nivas, Near RamDev Medical, Lake View Re...                 2
38  Fraghill (House Of Luxury Fragrances), Plot 60...                 2
68  Panchsheel Green 1, 39, Panchsheel Greens, Bha...                 2
41  GR Queens Pride, Koppa Rd, Suraksha Nagar, Yel...                 2
44  Greenwoods Extension Tower, Plot No. 1, Inform...           

***Insights :***
_______________
    * Shape of the dataset is 12,42,276 X 31
    * None of the columns is having unique values.
    * 'product_type', 'product_description', 'package_type', 'net_quantity', 'unit_price' are completely null columns.
    * 'brand_name' is a partialy null column with 170 null values.
    * Distinct store_id and store_name_location count are 125 and 100 respectively. While ivestigating, we found that 28 store_name_locations have exactly two store_ids each.
    


**5) Primary key Analysis**

As per the above informations, no column can be choose as primary key. 

In [10]:
# Check if the combination is unique
is_unique = not df.duplicated(subset=['unique_id', 'store_id','product_id_upc_ean', 'product_name']).any()

print("Is (unique_id, store_id, product_id_upc_ean, product_name) combination unique?", is_unique)

Is (unique_id, store_id, product_id_upc_ean, product_name) combination unique? False


Checked several combinations, but couldn't fix a unique combination

**6) Duplicate Analysis**

In [11]:
print("EXACT DUPLICATE ROWS")
print("--------------------------------")
exact_duplicate_count = df.duplicated().sum()
print("Exact duplicate rows:", exact_duplicate_count)


print("\nDUPLICATE PRODUCTS IN THE SAME STORE")
print("--------------------------------")
product_duplicate_count = df.duplicated(
    subset=['store_id', 'product_name']
).sum()

print(f"\nNumber of duplicate products (store_id + product_name): {product_duplicate_count}")


print("\n========== DUPLICATE ANALYSIS SUMMARY ==========")
print(f"Total Rows                  : {len(df)}")
print(f"Exact Duplicate Rows        : {exact_duplicate_count}")
print(f"Product Duplicate Rows      : {product_duplicate_count}")
print("================================================")

EXACT DUPLICATE ROWS
--------------------------------
Exact duplicate rows: 0

DUPLICATE PRODUCTS IN THE SAME STORE
--------------------------------

Number of duplicate products (store_id + product_name): 286078

========== DUPLICATE ANALYSIS SUMMARY ==========
Total Rows                  : 1177620
Exact Duplicate Rows        : 0
Product Duplicate Rows      : 286078


In [12]:
# checking for duplicate combinations of store_name_location, product_name, product_grammage, item_selling_price and package_type
dup_products = (
    df.groupby([
        'store_name_location',
        'product_name',
        'product_grammage',
        'item_selling_price',
        'package_type'
    ])
    .size()
    .reset_index(name='count')
)

dup_products = dup_products[dup_products['count'] > 1]\
                    .sort_values('count', ascending=False)

print("Duplicate combinations:", len(dup_products))
print(dup_products.head(20))

Duplicate combinations: 0
Empty DataFrame
Columns: [store_name_location, product_name, product_grammage, item_selling_price, package_type, count]
Index: []


In [43]:
# Check for products mapped to multiple categories
product_category_check = (
    df.groupby(['store_name_location', 'product_name','product_grammage', 'product_url'])['category']
      .nunique()
      .reset_index(name='category_count')
)

# Products mapped to more than one category
inconsistent_products = product_category_check[
    product_category_check['category_count'] > 1
].sort_values('category_count', ascending=False)

print("Number of inconsistent products:",
      len(inconsistent_products))

print(inconsistent_products.head())
print("\nNumber of inconsistent products:", len(inconsistent_products))

Number of inconsistent products: 84078
                                      store_name_location  \
49191   2, Mumbai Cantral Rd, Dadal Estate, Bane Compo...   
673562  RajMahal AC Mall, U-59, Puna Kumbariya Rd, Nea...   
474813  Hotel Raddisson, Ishwar Nagar, Gulmohar Colony...   
125900  7 Classes Hiranandani (Heera Panna Shopping Ce...   
408832  Erange Electronics, 266/2113, Motilal Nagar.1,...   

                                             product_name product_grammage  \
49191   Elicious Naturals Neem Leaf Powder For Hair, S...            200 g   
673562  Elicious Naturals Neem Leaf Powder For Hair, S...            200 g   
474813  Elicious Naturals Neem Leaf Powder For Hair, S...            200 g   
125900  Elicious Naturals Neem Leaf Powder For Hair, S...            200 g   
408832  Elicious Naturals Neem Leaf Powder For Hair, S...            200 g   

                                             product_url  category_count  
49191   https://www.swiggy.com/instamart/item/NJ8J

In [40]:
# Check for products mapped to multiple sub_categories
product_category_check = (
    df.groupby(['store_name_location', 'product_name','product_grammage'])['sub_category']
      .nunique()
      .reset_index(name='sub_category_count')
)

# Products mapped to more than one category
inconsistent_products = product_category_check[
    product_category_check['sub_category_count'] > 1
].sort_values('sub_category_count', ascending=False)

print("Number of inconsistent products:",
      len(inconsistent_products))

print(inconsistent_products.head())
print("\nNumber of inconsistent products:", len(inconsistent_products))

Number of inconsistent products: 179598
                                      store_name_location  \
760021  Scholars SPOT Girl's Hostel, 1, Niral Enem Hor...   
570022  Lodha Golf Course, Casa Bella, Casa Bella Gold...   
821257  Temple Gym, Near PD Plaza, Fazilpur Jharsa, Se...   
108354  45, Kodigehalli Main Rd, Amco Colony, Devinaga...   
899162  Vasant Vihar, 117-A, Vasant Vihar, Yojna-3, Jh...   

                                             product_name product_grammage  \
760021  Cadbury Celebrations Premium Selections Assort...            155 g   
570022  Cadbury Celebrations Premium Selections Assort...            155 g   
821257        Ferrero Rocher Moments Gift Pack (8 Pieces)          8 g x 2   
108354    Cadbury Dairy Milk Silk Pralines Chocolate Gift        264 g x 2   
899162  Cadbury Celebrations Assorted Chocolate Diwali...          119.2 g   

        sub_category_count  
760021                   5  
570022                   5  
821257                   4  
108354  

In [15]:
# Count distinct product names for each unique_id
product_name_counts = (
    df.groupby("unique_id")["product_name"]
      .nunique(dropna=False)
)

# unique_id values having more than one product_name
duplicate_unique_ids = product_name_counts[product_name_counts > 1].index

print(f"Number of unique_id values with different product_name: {len(duplicate_unique_ids)}")
print(duplicate_unique_ids.tolist())

Number of unique_id values with different product_name: 0
[]


***Insights :***
___________________

    * No exact duplicate columns exists on the dataset, but products duplicates on same stores
    * While deeply investigating, these duplicate products differ by grammage, price or package_type.
    * Also confirmed no products duplicates on same store under different categories.
    * In overall, the dataset is not containing duplicate products visibly.


7) MISSING VALUE ANALYSIS

As mentioned some columns are completely null, that not need to be evaluated. brand_name column is partially null with 170 null values. Lets evaluate that.

In [16]:
# Missing summary by store
print("\nMISSING SUMMARY BY STORE")
print("--------------------------------")
missing_by_store = df.groupby('store_name_location')['brand_name'].apply(
    lambda x: x.isnull().mean() * 100
).reset_index()

print(missing_by_store)



MISSING SUMMARY BY STORE
--------------------------------
                                  store_name_location  brand_name
0   0, New Kuldeep Nagar, Jodhewal, Ludhiana, Punj...    0.000000
1   126, MM Road, Pulikeshi Nagar, Bengaluru, Karn...    0.022638
2   15, LRN Colony, Hasthampatti, Salem, Tamil Nad...    0.022240
3   164, Bhairav Ji Cir, Sunny Nagar, Sanganer, Se...    0.009322
4   165/20, Gopal Mishra Rd, Bank Colony, Satin Se...    0.013411
..                                                ...         ...
95  Vasant Vihar, 117-A, Vasant Vihar, Yojna-3, Jh...    0.000000
96  Visakha Eye Hospital, #8-1-64, Nauka Nagar, Pe...    0.007671
97  Yadav Pan Shop, Survey No. 71 Front Of Gauri M...    0.015012
98  Yousuf Residency, RamNagar Gundu Rd, Bharat Na...    0.026396
99  Zepto Parent Shad, 656, 6th Cross Rd, 5th Phas...    0.005762

[100 rows x 2 columns]


In [17]:
# Missing summary by category
print("\nMISSING SUMMARY BY CATEGORY")
print("--------------------------------")
missing_by_category = df.groupby('category')['brand_name'].apply(
    lambda x: x.isnull().mean() * 100
).reset_index()

print(missing_by_category)


MISSING SUMMARY BY CATEGORY
--------------------------------
                          category  brand_name
0               Atta, Rice and Dal    0.125202
1                        Baby Care    0.000000
2                    Bath and Body    0.000711
3               Biscuits and Cakes    0.000000
4            Cereals and Breakfast    0.000000
5               Chips and Namkeens    0.000000
6                       Chocolates    0.000000
7           Cold Drinks and Juices    0.000000
8            Dairy, Bread and Eggs    0.001407
9                 Feminine Hygiene    0.000000
10                       Hair Care    0.000000
11  Ice Creams and Frozen Desserts    0.037439
12                          Makeup    0.082615
13                     Paan Corner    0.000000
14                        Skincare    0.001264
15                    Sweet Corner    0.000000


In [18]:
print("\nMISSING SUMMARY BY CITY")
print("--------------------------------")

missing_by_city = df.groupby('city')['brand_name'].apply(
    lambda x: x.isnull().mean() * 100
).reset_index()

print(missing_by_city)


MISSING SUMMARY BY CITY
--------------------------------
             city  brand_name
0       Ahmedabad    0.011579
1         Aligarh    0.000000
2       Bangalore    0.013625
3          Bhopal    0.010314
4     Bhubaneswar    0.000000
5       Chalakudy    0.018857
6      Chandigarh    0.008758
7         Chennai    0.016934
8      Coimbatore    0.007313
9           Delhi    0.026700
10     Ettumanoor    0.010553
11      Faridabad    0.009478
12      Ghaziabad    0.009232
13        Gurgaon    0.028005
14      Hyderabad    0.024626
15         Jaipur    0.017413
16    Kalamassery    0.017726
17     Karimnagar    0.008619
18          Kochi    0.008722
19        Kolkata    0.015960
20        Lucknow    0.000000
21       Ludhiana    0.000000
22         Mumbai    0.011408
23         Nagpur    0.000000
24    Navi Mumbai    0.025584
25      Prayagraj    0.000000
26           Pune    0.018629
27         Rajkot    0.018713
28         Ranchi    0.016055
29    S.A.S Nagar    0.011952
30          

***Insights :***
_______________

    * In store-wise, missing values are almost equally distributed.
    * In category-wise also mising percentage is below 0.2, which is negliglible.
    * In city-wise too, the missing percentage has no hike at any particular city.
    * So in overall missing% doesn't have any direct impact form store/category/city.

**8) Category Hierarchy Consistency**

In [19]:
print("\n DEPARTMENT - CATEGORY CONSISTENCY")
print("--------------------------------")

dept_category_map = df.groupby('department')['category'].unique()

for dept, cats in dept_category_map.items():
    print("\nDEPARTMENT:", dept)
    print(list(cats))



 DEPARTMENT - CATEGORY CONSISTENCY
--------------------------------

DEPARTMENT: Atta, Rice & Dal
['Atta, Rice and Dal']

DEPARTMENT: Baby Care
['Baby Care']

DEPARTMENT: Bakery & Biscuits
['Biscuits and Cakes']

DEPARTMENT: Beauty & Cosmetics
['Bath and Body', 'Makeup', 'Hair Care']

DEPARTMENT: Cold Drinks & Juices
['Cold Drinks and Juices']

DEPARTMENT: Dairy & Breakfast
['Cereals and Breakfast', 'Dairy, Bread and Eggs']

DEPARTMENT: Munchies
['Chips and Namkeens', 'Cereals and Breakfast']

DEPARTMENT: Personal Care
['Skincare', 'Feminine Hygiene', 'Hair Care', 'Bath and Body', 'Makeup']

DEPARTMENT: Sweet Tooth
['Ice Creams and Frozen Desserts', 'Sweet Corner', 'Chocolates', 'Paan Corner']


In [20]:
print("\n CATEGORY - SUBCATEGORY CONSISTENCY")
print("--------------------------------")

cat_subcat_map = df.groupby('category')['sub_category'].unique()

for cat, subcats in cat_subcat_map.items():
    print("\nCATEGORY:", cat)
    print(list(subcats))


 CATEGORY - SUBCATEGORY CONSISTENCY
--------------------------------

CATEGORY: Atta, Rice and Dal
['Besan, Sooji and Maida', 'Millets & Daliya', 'Other Flours', 'Toor, Moong and Urad', 'Rajma, Chola and Others', 'Atta', 'Poha & Puffed Rice', 'Rice']

CATEGORY: Baby Care
['Clothes & Accessories', 'Baby Food and Formula', 'Travel Needs & Baby Gears', 'Gifts & More', 'Books and Toys', 'Baby Wipes', 'Baby Bathing', 'Baby Diapers', 'Feeding and Teething Needs', 'Baby Pharma', 'Mom Care', 'Baby Hygiene', 'Baby Oral Care', 'Baby Oil and Talc', 'Baby Cream & Lotions']

CATEGORY: Bath and Body
['Body Lotion & Oils', 'Face Care', 'Fragrance & Talc', 'Mens Perfume', 'Womens Perfume', 'Bath & Beauty gifts', 'Lip care', 'Roll on', 'Shower gel', 'Multi groomers', 'Soaps', 'Bath Accessories', 'Oral care', 'Handwash']

CATEGORY: Biscuits and Cakes
['Baking Ingredients', 'Cookies', 'Marie & Digestive', 'Cream Biscuits', 'Gourmet collection', 'Healthy Snacking', 'Rusk', 'Wafers', 'Salted & Plain', 'Ca

***Insights :***
___________________

    * No specific inconsistencies noticed accross department-category or category-subcategory combinations

**9) Suspicious Value Checking**

In [21]:
def find_suspicious_values(df, column_name):
    suspicious_values = [
        'null', 'none', 'n/a', 'na', 'nil', 'not available',
        'not applicable', '-', '--', '0', ''
    ]
    
    mask = (
        df[column_name].isna() |
        df[column_name].astype(str).str.strip().str.lower().isin(suspicious_values) |
        df[column_name].str.fullmatch(r'\d+') |   # numeric only
        (df[column_name].str.len() < 3)    # too short
        )
    
    df_suspicious = df[mask]
    print(f"Number of suspicious values in '{column_name}': {len(df_suspicious)}")
    return df_suspicious[column_name].unique()

print(find_suspicious_values(df, 'product_name'))
print(find_suspicious_values(df, 'brand_name'))
print(find_suspicious_values(df, 'store_name_location'))
print(find_suspicious_values(df, 'department'))
print(find_suspicious_values(df, 'category'))
print(find_suspicious_values(df, 'sub_category'))

Number of suspicious values in 'product_name': 0
[]
Number of suspicious values in 'brand_name': 3348
['921' nan 'Yu' 'RG' 'Jo' 'SP' 'He' 'Go' 'P' 'SS' 'Lu' 'C4' 'Ob']
Number of suspicious values in 'store_name_location': 0
[]
Number of suspicious values in 'department': 0
[]
Number of suspicious values in 'category': 0
[]
Number of suspicious values in 'sub_category': 0
[]


***Insights :***
____________

    * Suspicious values noticed on brand_name column.
    * Checked website to confirm whether these brand_names really exists or not.
    * Most of them really exists, but urls of some brands are not able to access.
    * Complete urls on brand_name 'RG' and 'C4' and some urls on other brands are not accessible, which is a critical issue.

**10) Price Anomalies**

In [22]:
def check_negative_and_zero(df, column):
    """
    Returns rows where the given column has:
    - negative values (< 0)
    - zero values (== 0)
    """

    # Ensure numeric (safe for real-world messy data)
    df[column] = pd.to_numeric(df[column], errors='coerce')

    negative_values = df[df[column] < 0]
    zero_values = df[df[column] == 0]

    return {
        "negative_values": negative_values[['unique_id', column]],
        "zero_values": zero_values[['unique_id', column]]
    }

print("\nItem MRP:")
print(check_negative_and_zero(df, 'item_mrp'))
print("\nItem Selling Price:")
print(check_negative_and_zero(df, 'item_selling_price'))
print("\nProduct Price:")
print(check_negative_and_zero(df, 'product_price'))
print("\nPromo Offer Price:")
print(check_negative_and_zero(df, 'promo_offer_price'))


Item MRP:
{'negative_values': Empty DataFrame
Columns: [unique_id, item_mrp]
Index: [], 'zero_values': Empty DataFrame
Columns: [unique_id, item_mrp]
Index: []}

Item Selling Price:
{'negative_values': Empty DataFrame
Columns: [unique_id, item_selling_price]
Index: [], 'zero_values': Empty DataFrame
Columns: [unique_id, item_selling_price]
Index: []}

Product Price:
{'negative_values': Empty DataFrame
Columns: [unique_id, product_price]
Index: [], 'zero_values': Empty DataFrame
Columns: [unique_id, product_price]
Index: []}

Promo Offer Price:
{'negative_values': Empty DataFrame
Columns: [unique_id, promo_offer_price]
Index: [], 'zero_values': Empty DataFrame
Columns: [unique_id, promo_offer_price]
Index: []}


In [23]:
# check if item_selling_price is greater than item_mrp
df[df['item_selling_price'] > df['item_mrp']][['unique_id', 'item_selling_price', 'item_mrp']]

,unique_id,item_selling_price,item_mrp


In [24]:
# check if discount_in_percent is greater than 0 and item_selling_price is greater than or equal to item_mrp
df[
    (df['discount_in_percent'] > 0) &
    (df['item_selling_price'] >= df['item_mrp'])
][['unique_id', 'item_selling_price', 'item_mrp', 'discount_in_percent']]

,unique_id,item_selling_price,item_mrp,discount_in_percent


In [25]:
# check if promo_offer_description is not null and promo_offer_price is null
df[
    (df['promo_offer_description'].notnull()) &

    (df['promo_offer_price'].isnull())][['unique_id', 'promo_offer_description', 'promo_offer_price']]

,unique_id,promo_offer_description,promo_offer_price


In [26]:
# check if promo_offer_price is not null and promo_offer_price is not equal to item_selling_price
df[
    (df['promo_offer_price'].notnull()) &
    (df['promo_offer_price'] != df['item_selling_price'])
][['unique_id', 'promo_offer_description', 'promo_offer_price', 'item_selling_price']]

,unique_id,promo_offer_description,promo_offer_price,item_selling_price


In [27]:
# check if out_of_stock_flag is True and stock_count is greater than 0
df[
    (df['out_of_stock_flag'] == True) &
    (df['stock_count'] > 0)
][['unique_id', 'store_id', 'product_id_upc_ean']]

,unique_id,store_id,product_id_upc_ean


***Insights :***
____________

    * No specific anomalies found on price related columns
    * All such columns are greater than 0.
    * selling price and mrp are equal.
    * selling price is always less than mrp if discount percentage exists.
    * if promo offer description exists, promo offer price also exists.
    * If out of stock flag true, stock count is zero.

**11) URL Checking**


In [28]:
total_records = len(df)
missing_urls = df['product_url'].isnull().sum()
unique_urls = df['product_url'].nunique(dropna=True)
duplicate_urls = df['product_url'].duplicated().sum()

print("URL ANALYSIS SUMMARY")
print("-" * 40)
print(f"Total records           : {total_records}")
print(f"Missing URLs            : {missing_urls}")
print(f"Unique URLs             : {unique_urls}")
print(f"Duplicate URLs          : {duplicate_urls}")
print(f"Uniqueness %            : {(unique_urls/total_records)*100:.2f}%")

URL ANALYSIS SUMMARY
----------------------------------------
Total records           : 1177620
Missing URLs            : 0
Unique URLs             : 31263
Duplicate URLs          : 1146357
Uniqueness %            : 2.65%


In [29]:
# Duplicate URL Records
# ----------------------------
duplicate_url_df = df[
    df['product_url'].duplicated(keep=False)
].sort_values('product_url')

print("\nDuplicate URL Records:")
print(duplicate_url_df[['unique_id', 'product_name', 'product_url']].head())


Duplicate URL Records:
          unique_id                                       product_name  \
686358   4ODRST8F9Y  FACES CANADA Ultime Pro Splash Nail Enamel - P...   
1082629  PDQ82JR0MR  FACES CANADA Ultime Pro Splash Nail Enamel - P...   
196062   J2IC7YXXDK  FACES CANADA Ultime Pro Splash Nail Enamel - P...   
1024188  UVL2I4RVH9  FACES CANADA Ultime Pro Splash Nail Enamel - P...   
490548   SDLANFMT3S  FACES CANADA Ultime Pro Splash Nail Enamel - P...   

                                              product_url  
686358   https://www.swiggy.com/instamart/item/0009OYBIMZ  
1082629  https://www.swiggy.com/instamart/item/0009OYBIMZ  
196062   https://www.swiggy.com/instamart/item/0009OYBIMZ  
1024188  https://www.swiggy.com/instamart/item/0009OYBIMZ  
490548   https://www.swiggy.com/instamart/item/0009OYBIMZ  


In [30]:
# ----------------------------
# URL Format Validation
# ----------------------------

# Regex for HTTP/HTTPS URLs
url_pattern = re.compile(
    r'^(https?://)'               # http:// or https://
    r'([\w.-]+)'                  # domain
    r'(\.[a-zA-Z]{2,})'           # TLD
    r'([/\w .?%&=+-]*)?$'         # path/query
)

invalid_urls = df[
    df['product_url'].notnull() &
    ~df['product_url'].astype(str).str.match(url_pattern)
]

print("\nInvalid URL Format Count:", len(invalid_urls))

print("\nSample Invalid URLs:")
print(
    invalid_urls[
        ['unique_id', 'product_name', 'product_url']
    ].head(10)
)


Invalid URL Format Count: 0

Sample Invalid URLs:
Empty DataFrame
Columns: [unique_id, product_name, product_url]
Index: []


In [31]:
# ----------------------------
# URLs without http/https
# ----------------------------
missing_protocol = df[
    df['product_url'].notnull() &
    ~df['product_url'].astype(str).str.startswith(('http://', 'https://'))
]

print("\nURLs missing protocol (http/https):", len(missing_protocol))

print(
    missing_protocol[
        ['unique_id', 'product_name', 'product_url']
    ].head()
)


URLs missing protocol (http/https): 0
Empty DataFrame
Columns: [unique_id, product_name, product_url]
Index: []


***Insights***
___________
* Format and pattern of urls are correct
* But 1210545 duplicate urls present on the dataset.
* However these duplcates have different store location and different price range.

**12) City, Store, Pincode Analysis**

In [32]:
# Missing percentage by city

missing_by_city = (
    df.groupby('city')
      .apply(lambda x: x.isnull().mean().mean() * 100)
      .reset_index(name='missing_perc')
      .sort_values('missing_perc', ascending=False)
)

# Stockout percentage by city

df['out_of_stock_flag_num'] = (
    df['out_of_stock_flag']
    .astype(str)
    .str.lower()
    .isin(['true', 'yes', '1', 'y'])
    .astype(int)
)

stockout_by_city = (
    df.groupby('city')['out_of_stock_flag_num']
      .mean()
      .mul(100)
      .reset_index(name='stockout_perc')
      .sort_values('stockout_perc', ascending=False)
)

# Discount percentage by city

discount_by_city = (
    df.groupby('city')['discount_in_percent']
      .mean()
      .reset_index(name='avg_discount_perc')
      .sort_values('avg_discount_perc', ascending=False)
) 

# Number of unique products by city

products_by_city = (
    df.groupby('city')['unique_id']
      .nunique()
      .reset_index(name='product_count')
      .sort_values('product_count', ascending=False)
)

# Combine all metrics
city_analysis = (
    missing_by_city
    .merge(stockout_by_city, on='city', how='outer')
    .merge(discount_by_city, on='city', how='outer')
    .merge(products_by_city, on='city', how='outer')
    .sort_values('missing_perc', ascending=False)
)

print("\nCITY ANALYSIS SUMMARY")
print("--------------------------------")
print(city_analysis)

/tmp/ipykernel_7222/439306221.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.isnull().mean().mean() * 100)



CITY ANALYSIS SUMMARY
--------------------------------
             city  missing_perc  stockout_perc  avg_discount_perc  \
13        Gurgaon     16.129936      38.296383          15.928168   
32          Thane     16.129928      56.848047          15.397094   
35        Udaipur     16.129913      45.638133          15.886746   
9           Delhi     16.129894      45.632947          14.869773   
24    Navi Mumbai     16.129858      43.467508          14.505373   
14      Hyderabad     16.129827      46.076634          14.766627   
31          Surat     16.129812      39.742660          15.561844   
30          Salem     16.129750      47.381297          14.725453   
5       Chalakudy     16.129641      38.195361          16.183858   
27         Rajkot     16.129636      45.078593          16.140906   
26           Pune     16.129633      41.620741          15.305659   
16    Kalamassery     16.129604      49.596738          15.850394   
15         Jaipur     16.129594      33.932614 

In [33]:
# Missing percentage by store

missing_by_store = (
    df.groupby('store_name_location')
      .apply(lambda x: x.isnull().mean().mean() * 100)
      .reset_index(name='missing_perc')
      .sort_values('missing_perc', ascending=False)
)

# Stockout percentage by store

df['out_of_stock_flag_num'] = (
    df['out_of_stock_flag']
    .astype(str)
    .str.lower()
    .isin(['true', 'yes', '1', 'y'])
    .astype(int)
)

stockout_by_store = (
    df.groupby('store_name_location')['out_of_stock_flag_num']
      .mean()
      .mul(100)
      .reset_index(name='stockout_perc')
      .sort_values('stockout_perc', ascending=False)
)

# Discount percentage by store

discount_by_store = (
    df.groupby('store_name_location')['discount_in_percent']
      .mean()
      .reset_index(name='avg_discount_perc')
      .sort_values('avg_discount_perc', ascending=False)
) 

# Number of unique products by store

products_by_store = (
    df.groupby('store_name_location')['unique_id']
      .nunique()
      .reset_index(name='product_count')
      .sort_values('product_count', ascending=False)
)

# Combine all metrics
store_analysis = (
    missing_by_store
    .merge(stockout_by_store, on='store_name_location', how='outer')
    .merge(discount_by_store, on='store_name_location', how='outer')
    .merge(products_by_store, on='store_name_location', how='outer')
    .sort_values('missing_perc', ascending=False)
)

print("\nSTORE ANALYSIS SUMMARY")
print("--------------------------------")
print(store_analysis)

/tmp/ipykernel_7222/3222998141.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.isnull().mean().mean() * 100)



STORE ANALYSIS SUMMARY
--------------------------------
                                  store_name_location  missing_perc  \
82  Shubham Steel, B-60, MBR Enclave, Block B, Sec...     15.626339   
10  4, Raj Bhavan Rd, Raj Bhavan Quarters Colony, ...     15.626243   
86  Surya Mytri, Rd Number 3, Hanuman Nagar, Jubil...     15.626134   
83  Sohini IT Tech Park, Financial District, Nanak...     15.626042   
71  RajMahal AC Mall, U-59, Puna Kumbariya Rd, Nea...     15.626029   
..                                                ...           ...   
95  Vasant Vihar, 117-A, Vasant Vihar, Yojna-3, Jh...     15.625000   
16  Aarohi Twin Bungalows, Bopal, Ahmedabad, Gujar...     15.625000   
15  ABT Group, 1/C, 113 Sector 1, Gomti Nagar Exte...     15.625000   
5   2, Mumbai Cantral Rd, Dadal Estate, Bane Compo...     15.625000   
0   0, New Kuldeep Nagar, Jodhewal, Ludhiana, Punj...     15.625000   

    stockout_perc  avg_discount_perc  product_count  
82      38.536533          14.868759

In [34]:
# Missing percentage by pincode

missing_by_pincode = (
    df.groupby('pin_code')
      .apply(lambda x: x.isnull().mean().mean() * 100)
      .reset_index(name='missing_perc')
      .sort_values('missing_perc', ascending=False)
)

# Stockout percentage by pincode

df['out_of_stock_flag_num'] = (
    df['out_of_stock_flag']
    .astype(str)
    .str.lower()
    .isin(['true', 'yes', '1', 'y'])
    .astype(int)
)

stockout_by_pincode = (
    df.groupby('pin_code')['out_of_stock_flag_num']
      .mean()
      .mul(100)
      .reset_index(name='stockout_perc')
      .sort_values('stockout_perc', ascending=False)
)

# Discount percentage by pincode

discount_by_pincode = (
    df.groupby('pin_code')['discount_in_percent']
      .mean()
      .reset_index(name='avg_discount_perc')
      .sort_values('avg_discount_perc', ascending=False)
) 

# Number of unique products by pincode

products_by_pincode = (
    df.groupby('pin_code')['unique_id']
      .nunique()
      .reset_index(name='product_count')
      .sort_values('product_count', ascending=False)
)

# Combine all metrics
pincode_analysis = (
    missing_by_pincode
    .merge(stockout_by_pincode, on='pin_code', how='outer')
    .merge(discount_by_pincode, on='pin_code', how='outer')
    .merge(products_by_pincode, on='pin_code', how='outer')
    .sort_values('missing_perc', ascending=False)
)

print("\nPINCODE ANALYSIS SUMMARY")
print("--------------------------------")
print(pincode_analysis)

/tmp/ipykernel_7222/3770454122.py:5: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.isnull().mean().mean() * 100)



PINCODE ANALYSIS SUMMARY
--------------------------------
    pin_code  missing_perc  stockout_perc  avg_discount_perc  product_count
2     110075     15.626339      38.536533          14.868759           7559
55    500082     15.626243      61.000597          14.352994           8117
56    500084     15.626134      61.950998          14.407895           8949
49    500032     15.626042      45.496998          14.159328           7253
22    395010     15.626029      35.393567          14.940059           7275
..       ...           ...            ...                ...            ...
7     141007     15.625000      40.659020          16.585731           8298
84    680006     15.625000      38.508796          15.864191           8668
20    380058     15.625000      47.392893          15.705746           9696
43    440015     15.625000      27.046151          16.504757          10485
12    211019     15.625000      55.870044          16.681608           7289

[95 rows x 5 columns]


***Insights :***
__________________

    * missing percentage is almost equally distributed over all dimensions
    * stockout percentage and product count has significant hike based on some cities, stores, pincodes etc.
    * However discount_percentage is also have almost equal distribution over different dimensions

**13) High Price Variations**

In [44]:
# Price Variation Analysis
price_variation = (
    df.groupby('product_id_upc_ean')
      .agg(
          product_name=('product_name', 'first'),
          min_price=('item_selling_price', 'min'),
          max_price=('item_selling_price', 'max'),
          mean_price=('item_selling_price', 'mean'),
          num_locations=('city', 'nunique')
      )
      .reset_index()
)

# Calculate percentage variation
price_variation['price_variation_pct'] = (
    (price_variation['max_price'] - price_variation['min_price'])
    / price_variation['min_price'] * 100
).round(2)

# Flag products having >30% variation
high_variation = price_variation[
    price_variation['price_variation_pct'] > 30
].sort_values('price_variation_pct', ascending=False)

print(high_variation.head(20))

      product_id_upc_ean                                       product_name  \
21653         OJA86KOFWY                                       Campa Orange   
12298         DYU1NS9OBF                                 Nutella B ready T1   
28743         X1UP6TZ2CH   CLEAR Premium Drinking Water with Added Minerals   
27158         V5U7GYM8HO       RENEE Pink False Stick On Nails (Pack of 24)   
30422         Z0H3WCQT3U  NOICE Ganache Chocolate Cookies (Zero Palm Oil...   
10772         C78K0R32E5  NOICE Handmade Cashew Cookies (Zero Palm Oil B...   
5443          63FRSPU0EC     NOICE Natural Coconut Water (No Preservatives)   
21780         OOELHN2B5V                       Bisleri Mineral Water Bottle   
14481         GFJ23OWDLG  NOICE Handmade Coconut Cookies (Zero Palm Oil ...   
13028         ESWAQTN109             Himalayan Natural Mineral Water Bottle   
19573         M9QXZYJMLV     Red Bull Energy Drink - The Red Edition 250 ml   
14919         GY12SME90J  NOICE Coconut Water With M

In [45]:
# check if any product has more than one selling price with same grammage across different stores
price_variation = (
    df.groupby(['product_name', 'product_grammage'])['item_selling_price']
      .nunique()
      .reset_index(name='price_count')
)

# Keep only products having more than one selling price
price_variation = price_variation[
    price_variation['price_count'] > 1
]

print(price_variation)

                                            product_name product_grammage  \
3      100% Whole Wheat Bread 400GM & Pintola All Nat...          1 Combo   
5                                      10X Classic Besan             1 kg   
6                                      10X Classic Besan            500 g   
7                          10X Classic Chakki Fresh Atta            10 kg   
8                          10X Classic Chakki Fresh Atta             5 kg   
...                                                  ...              ...   
37541  plum coconut milk & peptides strength & shine ...          1 combo   
37542  plum green tea alcohol free toner and cleansin...          1 combo   
37543  plum green tea oil free moisturiser and free t...          1 combo   
37545                      skin & strands Glycerine Soap            125 g   
37557                               uncle john Pistachio           450 ml   

       price_count  
3                4  
5                6  
6           

In [46]:
# Group by product_name and product_grammage and find price varation statistics
price_analysis = (
    df.groupby(['product_name', 'product_grammage'])['item_selling_price']
      .agg(['count', 'min', 'max', 'mean', 'std'])
      .reset_index()
)

# Calculate price difference and percentage variation
price_analysis['price_diff'] = (
    price_analysis['max'] - price_analysis['min']
)

price_analysis['pct_variation'] = (
    (price_analysis['price_diff'] / price_analysis['mean']) * 100
).round(2)

# Flag products with >20% variation
high_variation = price_analysis[
    price_analysis['pct_variation'] > 20
].sort_values('pct_variation', ascending=False)

print(high_variation)

                                            product_name product_grammage  \
6198                                        Campa Orange       500 ml x 4   
1925                              Amul Shrikhand Elaichi            500 g   
13675  HealthFields Organic Rozana Toor Split (Arhar ...             1 kg   
31676             Stayfree Tampons by OB Reg 10's Carton        10 pieces   
35974  Veet Ready-to-use Wax Strips (Normal Skin) (8 ...         8 pieces   
...                                                  ...              ...   
4150                           Bikaji Gol-Matol Rasgulla          1.25 kg   
16268  Khadi Natural Neem Tulsi Handmade Soap | Remov...            375 g   
32216                          Supreme Harvest Chana Dal             1 kg   
18341  Loacker Quadratini Chocolate Bite Size Wafer C...            125 g   
19674          Mamaearth Vitamin C Daily Glow Face Cream             80 g   

       count  min   max        mean         std  price_diff  pct_variation 

In [47]:
# check if any product has more than one selling price with same grammage on same store
price_variation = (
    df.groupby(['store_name_location', 'product_name', 'product_grammage'])['item_selling_price']
      .agg(['min', 'max', 'nunique', 'count'])
      .reset_index()
)

# Keep only products having more than one unique selling price
price_variation = price_variation[
    price_variation['nunique'] > 1
]

# Calculate price difference
price_variation['price_difference'] = (
    price_variation['max'] - price_variation['min']
)

print(price_variation.sort_values('price_difference', ascending=False))

                                      store_name_location  \
438695  Ganpati Book Shop, Shop No. 22, Stadium Road, ...   
619989  New U, Cinepolis, Shaheed Path, Gomti Nagar, L...   
846412  The School Of Legal Illuminati(sli), Plot No 7...   
629272  Padamughal Metro Station, Palarivattom-Kakkana...   
882674  UniQ Technologies, 74, 2nd St, Opp. Cristal Ic...   
...                                                   ...   
181835  Akash Ganga Building, Plot No.17, 18, 19 100 F...   
181273  Akash Ganga Building, Plot No.17, 18, 19 100 F...   
209347  Alackrity Consols Private Limited, Venkateshwa...   
403972  Early World International Pre-School, 91, Gree...   
540217  Juhu Gaon Marg, Juhu Nagar, Juhu Village, Sect...   

                                             product_name product_grammage  \
438695  Matrix Mega Smooth Anti-Frizz Combo with Shea ...          1 combo   
619989  Matrix Mega Smooth Anti-Frizz Combo with Shea ...          1 combo   
846412  Matrix Mega Smooth Anti-F

***Insights :***
_________________

    * Found price anomalies with high variation of price for the same product with the same grammage unit.
    * Multiple products are there with the same product_name, grammage, store_name, and different offer price.
    * The product_url of such products are too same and not accessible.